# 03 — İngilizce Ad Köprüsü ve DDInter Eşlemesi (İP3 - Kişi A & Kişi B)

Bu çalışma kitabı, 2. Hafta Görev Dağılımı ve Veri Birleştirme Planı uyarınca iki bölümden oluşmaktadır:
- **Bölüm 1 (Hücre 0 – 16 — Kişi A):** SKRS ATC Sözlüğü, DDInter İlaç Kataloğu (`ddinter_ilac.csv`), Yol A (Tek bileşenli ürünler) ve Yol B (Kombine ürünler).
- **Bölüm 2 (Hücre 17 – 25 — Kişi B):** SKRS Adımı (Kural 5/6), Yol C (Kural tabanlı çeviri), DDInter eşleme (`ddinter_id`), PubChem kontrolü, Kişi C için `ruhsatli_esleme_taslagi.csv`, nihai `etken_madde.csv` ve 4 zorunlu tablodan oluşan `docs/esleme_raporu.md`.


In [1]:
import sys
import glob
from pathlib import Path
import re
from collections import Counter
from typing import Optional, Union
import pandas as pd
import numpy as np
from rapidfuzz import fuzz

# Proje kök dizinini ekle
kok = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(kok))

from src.normalize import ad_normalize, eslesme_anahtari

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)

print("✓ Ortam ve kütüphaneler başarıyla yüklendi.")
print(f"  Proje kökü: {kok}")


✓ Ortam ve kütüphaneler başarıyla yüklendi.
  Proje kökü: /Users/reyhanbuseyanar/Desktop/ddi-final


In [2]:
# Ara veri dosyalarının yüklenmesi
urun_yol = kok / 'data' / 'interim' / 'urun.csv'
ue_yol = kok / 'data' / 'interim' / 'urun_etken.csv'
skrs_yol = kok / 'data' / 'raw' / 'skrs_2026-10-14.xlsx'
ddinter_yollar = sorted(glob.glob(str(kok / 'data' / 'raw' / 'ddinter_code_*.csv')))

df_urun = pd.read_csv(urun_yol, dtype=str)
df_urun['atc_celiski'] = pd.to_numeric(df_urun['atc_celiski'], errors='coerce').fillna(0).astype(int)
df_urun['kapsam_disi'] = pd.to_numeric(df_urun['kapsam_disi'], errors='coerce').fillna(0).astype(int)
df_urun['barkod_gecersiz'] = pd.to_numeric(df_urun['barkod_gecersiz'], errors='coerce').fillna(0).astype(int)

df_ue = pd.read_csv(ue_yol, dtype=str)
df_ue['sira'] = pd.to_numeric(df_ue['sira'], errors='coerce').fillna(1).astype(int)

print("✓ Veri setleri yüklendi:")
print(f"  urun.csv satır sayısı         : {len(df_urun):,}")
print(f"  urun_etken.csv satır sayısı   : {len(df_ue):,}")
print(f"  Benzersiz Türkçe bileşen sayısı: {df_ue['bilesen_tr'].nunique():,}")
print(f"  Bulunan DDInter dosyası sayısı: {len(ddinter_yollar)}")


✓ Veri setleri yüklendi:
  urun.csv satır sayısı         : 23,801
  urun_etken.csv satır sayısı   : 32,204
  Benzersiz Türkçe bileşen sayısı: 3,111
  Bulunan DDInter dosyası sayısı: 8


## 1. SKRS ATC Adı Sözlüğünün Oluşturulması

SKRS'in tüm satırlarından (aktif + pasif) `ATC kodu → ATC adı` sözlüğü çıkarılır.  
- Birden fazla ad geçen ATC kodları için en sık geçen ad (`mode()`) temel alınır.
- Resmi SKRS ATC adı olduğu gibi korunur; keyfi parantez temizliği yapılmaz, şüpheli durumlar `docs/acik_sorular.md` dosyasına kaydedilir.


In [4]:
def _temizle_atc_adi(atc_adi: Optional[str]) -> Optional[str]:
    """ATC adından görünmez karakterleri ve gereksiz boşlukları temizler.
    
    Not: PDF sabit kuralları gereği parantez içi ifadeler (vit c, human vb.)
    keyfi olarak silinmez, resmi SKRS ATC adı korunur. Şüpheli durumlar
    docs/acik_sorular.md dosyasına yazılır.
    """
    if pd.isna(atc_adi):
        return None
    s = str(atc_adi).strip()
    if not s or s.lower() == "nan":
        return None
    s = re.sub(r"\s+", " ", s).strip()
    return s or None


def skrs_atc_sozlugu_olustur(skrs_yolu: Union[str, Path]) -> dict[str, str]:
    """SKRS'in aktif ve pasif sekmelerinden 'ATC kodu → ATC adı' sözlüğü çıkarır.

    Birden fazla ad geçen kodlar için en sık geçen (mode) ad alınır.
    """
    skrs_p = Path(skrs_yolu)
    if not skrs_p.exists():
        raise FileNotFoundError(f"SKRS dosyası bulunamadı: {skrs_p}")

    df_aktif = pd.read_excel(skrs_p, sheet_name="AKTİF ÜRÜNLER LİSTESİ", header=2, dtype=str)
    df_pasif = pd.read_excel(skrs_p, sheet_name="PASİF ÜRÜNLER LİSTESİ", header=2, dtype=str)
    df_skrs = pd.concat([df_aktif, df_pasif], ignore_index=True)

    df_skrs["ATC Kodu"] = df_skrs["ATC Kodu"].astype(str).str.strip().str.upper()
    df_skrs["ATC Adı Temiz"] = df_skrs["ATC Adı"].apply(_temizle_atc_adi)

    gecerli = df_skrs[
        df_skrs["ATC Kodu"].notna()
        & (df_skrs["ATC Kodu"] != "")
        & (df_skrs["ATC Kodu"] != "NAN")
        & df_skrs["ATC Adı Temiz"].notna()
        & (df_skrs["ATC Adı Temiz"] != "")
    ]

    sozluk = (
        gecerli.groupby("ATC Kodu")["ATC Adı Temiz"]
        .agg(lambda x: x.mode().iloc[0] if not x.empty else "")
        .to_dict()
    )
    return sozluk


# Sözlüğü oluştur
atc_sozlugu = skrs_atc_sozlugu_olustur(skrs_yol)

print(f"✓ SKRS ATC Sözlüğü oluşturuldu:")
print(f"  Sözlükteki benzersiz ATC kodu sayısı: {len(atc_sozlugu):,}")

# Örnek sözlük kayıtları
ornek_kodlar = ['M01AE17', 'B01AA03', 'J01FA09', 'C09DB01', 'A10BK03', 'N02BE01', 'A11GA01']
print()
print("Örnek Sözlük Eşleşmeleri:")
for kod in ornek_kodlar:
    print(f"  {kod:<8} -> {atc_sozlugu.get(kod, 'BULUNAMADI')}")


✓ SKRS ATC Sözlüğü oluşturuldu:
  Sözlükteki benzersiz ATC kodu sayısı: 1,795

Örnek Sözlük Eşleşmeleri:
  M01AE17  -> dexketoprofen
  B01AA03  -> warfarin
  J01FA09  -> clarithromycin
  C09DB01  -> valsartan and amlodipine
  A10BK03  -> empagliflozin
  N02BE01  -> paracetamol
  A11GA01  -> ascorbic acid (vit c)


## 2. DDInter İlaç Tablosunun Çıkarılması (`ddinter_ilac.csv`)

Adım 4'ün ilk maddesi gereği 8 DDInter dosyasından (`code_A` ... `code_V`) benzersiz ilaç tablosu çıkarılır:
- `DDInterID_A, Drug_A` ve `DDInterID_B, Drug_B` alt alta eklenip `ad_normalize()` uygulandıktan sonra tekrarlar atılır.
- Çıktı: `data/interim/ddinter_ilac.csv` (~1.939 satır).


In [6]:
def ddinter_ilac_tablosu_olustur(ddinter_dosyalari: list[Union[str, Path]]) -> pd.DataFrame:
    """8 DDInter CSV dosyasından benzersiz ilaç tablosunu çıkarır.

    DDInterID_A, Drug_A ve DDInterID_B, Drug_B alt alta eklenir, tekrarlar atılır.
    Ad normalizasyonu ad_normalize() ile yapılır.
    Beklenen satır sayısı: ~1.939 satır.
    """
    records = []
    for f in ddinter_dosyalari:
        p = Path(f)
        if not p.exists():
            continue
        df = pd.read_csv(p)
        part_a = df[["DDInterID_A", "Drug_A"]].rename(
            columns={"DDInterID_A": "ddinter_id", "Drug_A": "drug_name"}
        )
        part_b = df[["DDInterID_B", "Drug_B"]].rename(
            columns={"DDInterID_B": "ddinter_id", "Drug_B": "drug_name"}
        )
        records.append(part_a)
        records.append(part_b)

    df_all = pd.concat(records, ignore_index=True)
    df_all["ad_norm"] = df_all["drug_name"].apply(ad_normalize)
    df_all = df_all.drop_duplicates(subset=["ddinter_id", "ad_norm"])
    df_all["_num"] = df_all["ddinter_id"].apply(lambda x: int(re.search(r"\d+", str(x)).group()) if re.search(r"\d+", str(x)) else 0)
    df_all = df_all.sort_values("_num").drop(columns=["_num"]).reset_index(drop=True)
    return df_all[["ddinter_id", "drug_name", "ad_norm"]]


df_ddinter_ilac = ddinter_ilac_tablosu_olustur(ddinter_yollar)

ddinter_cikti_yol = kok / 'data' / 'interim' / 'ddinter_ilac.csv'
df_ddinter_ilac.to_csv(ddinter_cikti_yol, index=False)

print(f"✓ ddinter_ilac.csv başarıyla kaydedildi: {ddinter_cikti_yol}")
print(f"  Satır sayısı: {len(df_ddinter_ilac):,}")
print(f"  Benzersiz ddinter_id sayısı: {df_ddinter_ilac['ddinter_id'].nunique():,}")
print(f"  Benzersiz ad_norm sayısı   : {df_ddinter_ilac['ad_norm'].nunique():,}")

print()
print("İlk 5 DDInter İlaç Kaydı:")
print(df_ddinter_ilac.head())


✓ ddinter_ilac.csv başarıyla kaydedildi: /Users/reyhanbuseyanar/Desktop/ddi-final/data/interim/ddinter_ilac.csv
  Satır sayısı: 1,939
  Benzersiz ddinter_id sayısı: 1,939
  Benzersiz ad_norm sayısı   : 1,939

İlk 5 DDInter İlaç Kaydı:
  ddinter_id             drug_name               ad_norm
0   DDInter1              Abacavir              abacavir
1   DDInter2         Abaloparatide         abaloparatide
2   DDInter3  Abametapir (topical)  abametapir (topical)
3   DDInter4              Abarelix              abarelix
4   DDInter5             Abatacept             abatacept


## 3. Adım 3 — Yol A: Tek Bileşenli Ürünler

**Kural:**
- Tek bileşenli ürün (`bilesen_sayisi == 1`), 7 haneli ATC kodu, `atc_celiski == 0` ve `kaynak == 'Ruhsatlı'`.
- ATC adı SKRS sözlüğünden doğrudan İngilizce karşılık olarak alınır (`deksketoprofen` → `dexketoprofen`).
- Genel kombinasyon ifadeleri (`combinations`, `excl.`, `various`) içeren adlar elenir.
- Bir bileşen birden fazla tek bileşenli üründe geçiyorsa en çok oy alan (mode) ATC/ad çifti seçilir.


In [8]:
_KOMBINASYON_REGEX = re.compile(r"\bcombinations\b|excl\.|\bvarious\b", re.IGNORECASE)


def atc_kombinasyon_mu(atc_adi: Optional[str]) -> bool:
    """ATC adının 'combinations', 'excl.' veya 'various' içerip içermediğini kontrol eder.
    
    Not: 'excl.' doğrudan nokta ile aranır; 'excluding' gibi sözcüklerle karışmaz.
    """
    if not atc_adi:
        return False
    return bool(_KOMBINASYON_REGEX.search(str(atc_adi)))


def yol_a_esle(
    df_urun: pd.DataFrame,
    df_urun_etken: pd.DataFrame,
    atc_sozlugu: dict[str, str],
) -> pd.DataFrame:
    """Yol A eşleştirmesi: Tek bileşenli, 7 haneli ATC'li, atc_celiski=0 olan ürünler.

    PDF Sayfa 4 Kuralı: ATC adı doğrudan İngilizce addır (deksketoprofen → dexketoprofen).
    Tek bileşenli ürünlerde doğrudan atama yapılır, bulanık eşleme (rapidfuzz) KULLANILMAZ.
    Birden fazla üründe geçen bileşenler için en sık geçen (mode) ATC/ad çifti seçilir.
    """
    bilesen_sayisi = df_urun_etken.groupby("barkod")["bilesen_tr"].count().to_dict()

    mask = (
        df_urun["barkod"].map(bilesen_sayisi).eq(1)
        & df_urun["atc_final"].astype(str).str.len().eq(7)
        & df_urun["atc_celiski"].astype(str).isin(["0", "0.0", 0])
        & df_urun["kaynak"].eq("Ruhsatlı")
    )
    yol_a_urunler = df_urun[mask].copy()

    yol_a_urunler["atc_adi"] = yol_a_urunler["atc_final"].map(atc_sozlugu)
    yol_a_urunler = yol_a_urunler[
        yol_a_urunler["atc_adi"].notna() & (yol_a_urunler["atc_adi"] != "")
    ]
    yol_a_urunler = yol_a_urunler[~yol_a_urunler["atc_adi"].apply(atc_kombinasyon_mu)].copy()

    m = df_urun_etken.merge(
        yol_a_urunler[["barkod", "atc_final", "atc_adi"]], on="barkod", how="inner"
    )
    m["ad_en"] = m["atc_adi"].apply(ad_normalize)

    sonuclar = []
    for b_tr, grp in m.groupby("bilesen_tr"):
        most_common_pair = Counter(zip(grp["ad_en"], grp["atc_final"])).most_common(1)[0][0]
        sonuclar.append({
            "bilesen_tr": b_tr,
            "ad_en": most_common_pair[0],
            "atc": most_common_pair[1],
            "yol": "A",
        })

    return pd.DataFrame(sonuclar)


df_yol_a = yol_a_esle(df_urun, df_ue, atc_sozlugu)

print(f"✓ Yol A eşleştirmesi tamamlandı:")
print(f"  Yol A ile eşleşen benzersiz bileşen sayısı: {len(df_yol_a):,}")

print()
print("Yol A Örnek Eşleşmeler:")
print(df_yol_a[['bilesen_tr', 'ad_en', 'atc']].head(10))


✓ Yol A eşleştirmesi tamamlandı:
  Yol A ile eşleşen benzersiz bileşen sayısı: 1,922

Yol A Örnek Eşleşmeler:
                                          bilesen_tr                                              ad_en      atc
0                            % 20'lik insan albumini                                            albumin  B05AA01
1  % 25 insulin lispro cozelti, % 75 protamin sus...                                     insulin lispro  A10AD04
2  % 25 insulin lispro, % 75 insulin lispro protamin                                     insulin lispro  A10AD04
3                            % 25'lik insan albumini                                            albumin  B05AA01
4  % 50 insulin lispro cozelti, % 50 protamin sus...                                     insulin lispro  A10AD04
5  % 50 insulin lispro, % 50 insulin lispro protamin                                     insulin lispro  A10AD04
6   %25 insulin lispro / %75 insulin lispro protamin                                     insulin li

## 4. Adım 3 — Yol B: Kombine Ürünler

**Kural:**
- Kombine ürünler (`bilesen_sayisi > 1`), 7 haneli ATC, `atc_celiski == 0` ve `kaynak == 'Ruhsatlı'`.
- ATC adında `combinations`, `excl.` veya `various` varsa bölünmez (Yol C ve D'ye bırakılır).
- ATC adı `and` ve `,` ile bölünür. Parça sayısı bileşen sayısına eşitse `rapidfuzz` ile eşleştirme yapılır (benzerlik skoru $\ge 80$).
- Yol A'da eşleşen bileşenler mükerrer eşlemeyi önlemek için hariç tutulur.


In [10]:
def atc_adi_parcala(atc_adi: str) -> list[str]:
    """ATC adını 'and' ve ',' ile parçalara ayırır."""
    if not atc_adi:
        return []
    parts = [p.strip() for p in re.split(r"\s+and\s+|,", str(atc_adi), flags=re.IGNORECASE) if p.strip()]
    return parts


def yol_b_esle(
    df_urun: pd.DataFrame,
    df_urun_etken: pd.DataFrame,
    atc_sozlugu: dict[str, str],
    onceki_eslesmeler: Optional[set[str]] = None,
) -> pd.DataFrame:
    """Yol B eşleştirmesi: Kombine ürünlerde ATC adını parçalayarak rapidfuzz ile eşleme.

    Kural:
    - Kombine ürün (bilesen_sayisi > 1), 7 haneli ATC, atc_celiski=0, kaynak=Ruhsatlı.
    - ATC adı 'combinations', 'excl.' veya 'various' içeriyorsa bölünmez (Yol C/D'ye kalır).
    - Parça sayısı bileşen sayısına eşit olmalıdır.
    - Parçalar rapidfuzz benzerlik skoru >= 80 ile eşlenir.
    - Yol A'da eşleşenler hariç tutulur.
    """
    onceki = set(onceki_eslesmeler) if onceki_eslesmeler else set()

    ue_grouped = (
        df_urun_etken.sort_values(["barkod", "sira"])
        .groupby("barkod")["bilesen_tr"]
        .apply(list)
        .to_dict()
    )

    mask = (
        df_urun["barkod"].map(lambda b: len(ue_grouped.get(b, []))).gt(1)
        & df_urun["atc_final"].astype(str).str.len().eq(7)
        & df_urun["atc_celiski"].astype(str).isin(["0", "0.0", 0])
        & df_urun["kaynak"].eq("Ruhsatlı")
    )
    kombine_urunler = df_urun[mask].copy()

    kombine_urunler["atc_adi"] = kombine_urunler["atc_final"].map(atc_sozlugu)
    kombine_urunler = kombine_urunler[
        kombine_urunler["atc_adi"].notna() & (kombine_urunler["atc_adi"] != "")
    ]
    kombine_urunler = kombine_urunler[~kombine_urunler["atc_adi"].apply(atc_kombinasyon_mu)].copy()

    adaylar = []
    for _, row in kombine_urunler.iterrows():
        barkod = row["barkod"]
        atc = row["atc_final"]
        atc_adi = row["atc_adi"]
        tr_bilesenler = ue_grouped.get(barkod, [])

        parts = atc_adi_parcala(atc_adi)
        if len(parts) != len(tr_bilesenler):
            continue

        used_parts = set()
        prod_matches = []
        for tr in tr_bilesenler:
            best_p = None
            best_score = -1
            best_idx = -1
            for idx, p in enumerate(parts):
                if idx in used_parts:
                    continue
                score = fuzz.ratio(eslesme_anahtari(tr), ad_normalize(p))
                if score > best_score:
                    best_score = score
                    best_p = p
                    best_idx = idx

            if best_score >= 80 and best_p is not None:
                used_parts.add(best_idx)
                prod_matches.append((tr, ad_normalize(best_p), atc, best_score))

        if len(prod_matches) == len(tr_bilesenler):
            adaylar.extend(prod_matches)

    yol_b_dict = {}
    for tr, ad_en, atc, score in adaylar:
        if tr in onceki:
            continue
        if tr not in yol_b_dict or score > yol_b_dict[tr]["eslesme_skoru"]:
            yol_b_dict[tr] = {
                "bilesen_tr": tr,
                "ad_en": ad_en,
                "atc": atc,
                "yol": "B",
                "eslesme_skoru": score,
            }

    return pd.DataFrame(list(yol_b_dict.values()))


eslesenler_a = set(df_yol_a['bilesen_tr'])
df_yol_b = yol_b_esle(df_urun, df_ue, atc_sozlugu, onceki_eslesmeler=eslesenler_a)

print(f"✓ Yol B eşleştirmesi tamamlandı:")
print(f"  Yol B ile YENİ eşleşen benzersiz bileşen sayısı: {len(df_yol_b):,}")

print()
print("Yol B Örnek Eşleşmeler:")
print(df_yol_b[['bilesen_tr', 'ad_en', 'atc', 'eslesme_skoru']].head(10))


✓ Yol B eşleştirmesi tamamlandı:
  Yol B ile YENİ eşleşen benzersiz bileşen sayısı: 66

Yol B Örnek Eşleşmeler:
             bilesen_tr                  ad_en      atc  eslesme_skoru
0            sakubitril             sacubitril  C09DX04      90.000000
1            trastuzuma            trastuzumab  L01FY01      95.238095
2       etinilestradiol       ethinylestradiol  G03AA12      90.322581
3           fluticasone            fluticasone  R03AK06     100.000000
4           biktegravir            bictegravir  J05AR20      90.909091
5          emtrisitabin          emtricitabine  J05AR20      88.000000
6  tenofovir alefenamid  tenofovir alafenamide  J05AR20      92.682927
7             lopinavir              lopinavir  J05AR10     100.000000
8           velpatasvir            velpatasvir  J05AP56     100.000000
9         voksilaprevir           voxilaprevir  J05AP56      88.000000


## 5. `etken_madde.csv` (Yol A/B) Taslağının Oluşturulması ve Kaydedilmesi

Tüm 3.111 benzersiz Türkçe bileşen için:
- `urun_sayisi`: Bu bileşeni içeren benzersiz barkod / ürün sayısı hesaplanır.
- Yol A ve Yol B eşleşmeleri birleştirilir.
- Eşleşmeyen tüm bileşenler (SKRS ürünleri dahil) `yol='eslesmedi'` olarak Kişi B'ye (Yol C, Sabit Kural 5 ve Yol D için) devredilir.
- Tablo `urun_sayisi` azalan sırada sıralanarak kaydedilir.


In [12]:
def etken_madde_olustur(
    df_urun: pd.DataFrame,
    df_urun_etken: pd.DataFrame,
    atc_sozlugu: dict[str, str],
) -> pd.DataFrame:
    """Yol A ve Yol B eşlemelerini birleştirerek etken_madde.csv (Yol A/B) taslağını üretir.

    Kişi A Görev Sınırı:
    - Sadece Yol A ve Yol B uygulanır.
    - Yol A ve Yol B ile eşleşmeyen tüm bileşenler (SKRS dahil) yol='eslesmedi' olarak
      Kişi B'ye (Yol C, Kural 5 ve Yol D için) devredilir.
    - Toplam satır sayısı tam 3.111 benzersiz bileşendir.
    - urun_sayisi (ürün sayısı) azalan sırada dizilir.
    """
    urun_sayisi_seri = df_urun_etken.groupby("bilesen_tr")["barkod"].nunique()
    tum_bilesenler = pd.DataFrame({
        "bilesen_tr": urun_sayisi_seri.index,
        "urun_sayisi": urun_sayisi_seri.values,
    })

    # 1. Yol A
    df_yol_a = yol_a_esle(df_urun, df_urun_etken, atc_sozlugu)
    eslesenler_a = set(df_yol_a["bilesen_tr"])

    # 2. Yol B (Yol A haricindekiler)
    df_yol_b = yol_b_esle(df_urun, df_urun_etken, atc_sozlugu, onceki_eslesmeler=eslesenler_a)

    # 3. YALNIZCA Yol A ve Yol B birleştirilir (SKRS kaldırıldı!)
    birlestirilen = pd.concat([df_yol_a, df_yol_b], ignore_index=True)

    # 4. Tüm bileşenlerle sol birleştirme (left join)
    sonuc = tum_bilesenler.merge(
        birlestirilen[["bilesen_tr", "ad_en", "atc", "yol"]],
        on="bilesen_tr",
        how="left",
    )

    # Yol A ve B'de çözülemeyenler Kişi B'ye 'eslesmedi' olarak devredilir
    sonuc["yol"] = sonuc["yol"].fillna("eslesmedi")

    # Sütun sırası: bilesen_tr, ad_en, atc, yol, urun_sayisi
    sonuc = sonuc[["bilesen_tr", "ad_en", "atc", "yol", "urun_sayisi"]]
    sonuc = sonuc.sort_values(by=["urun_sayisi", "bilesen_tr"], ascending=[False, True]).reset_index(drop=True)

    return sonuc


df_etken_madde = etken_madde_olustur(df_urun, df_ue, atc_sozlugu)

em_cikti_yol = kok / 'data' / 'interim' / 'etken_madde.csv'
df_etken_madde.to_csv(em_cikti_yol, index=False)

print(f"✓ etken_madde.csv (Yol A/B taslağı) başarıyla kaydedildi: {em_cikti_yol}")
print(f"  Toplam benzersiz bileşen sayısı: {len(df_etken_madde):,}")
print()
print("Yol Dağılımı:")
print(df_etken_madde['yol'].value_counts())

print()
print("En Sık Kullanılan İlk 15 Bileşen:")
print(df_etken_madde.head(15))


✓ etken_madde.csv (Yol A/B taslağı) başarıyla kaydedildi: /Users/reyhanbuseyanar/Desktop/ddi-final/data/interim/etken_madde.csv
  Toplam benzersiz bileşen sayısı: 3,111

Yol Dağılımı:
yol
A            1922
eslesmedi    1123
B              66
Name: count, dtype: int64

En Sık Kullanılan İlk 15 Bileşen:
                                  bilesen_tr               ad_en      atc        yol  urun_sayisi
0                              sodyum klorur     sodium chloride  B05XA03          A         1059
1                                   dekstroz       carbohydrates  B05BA03          A          720
2                            kalsiyum klorur                 NaN      NaN  eslesmedi          400
3                            potasyum klorur  potassium chloride  B05XA01          A          387
4                          hidroklorotiyazid                 NaN      NaN  eslesmedi          379
5                                parasetamol         paracetamol  N02BE01          A          377
6          

## 6. DDInter Tam Ad Eşleşme Ön Testi

Yol A ve Yol B ile üretilen İngilizce adların (`ad_en`) DDInter veritabanında (`ddinter_ilac.csv`) birebir tam eşleşme durumunun kontrolü.


In [14]:
ddinter_dict = df_ddinter_ilac.set_index('ad_norm')['ddinter_id'].to_dict()

# Eşleşenlerin DDInter karşılığı
df_test = df_etken_madde.copy()
df_test['ddinter_id'] = df_test['ad_en'].map(ddinter_dict)

print("DDInter Tam Ad Eşleşme Durumu (Yol Bazında):")
for y in ['A', 'B', 'eslesmedi']:
    sub = df_test[df_test['yol'] == y]
    tam_eslesen = sub['ddinter_id'].notna().sum()
    toplam = len(sub)
    oran = (tam_eslesen / toplam * 100) if toplam > 0 else 0
    print(f"  Yol {y:<9}: Toplam = {toplam:>4}, DDInter'da Tam Eşleşen = {tam_eslesen:>4} (%{oran:5.1f})")

toplam_tam_eslesen = df_test['ddinter_id'].notna().sum()
print()
print(f"Toplam DDInter'da Tam Adı Bulunan Bileşen Sayısı: {toplam_tam_eslesen:,} / {len(df_test):,} (%{toplam_tam_eslesen/len(df_test)*100:.1f})")


DDInter Tam Ad Eşleşme Durumu (Yol Bazında):
  Yol A        : Toplam = 1922, DDInter'da Tam Eşleşen = 1197 (% 62.3)
  Yol B        : Toplam =   66, DDInter'da Tam Eşleşen =   32 (% 48.5)
  Yol eslesmedi: Toplam = 1123, DDInter'da Tam Eşleşen =    0 (%  0.0)

Toplam DDInter'da Tam Adı Bulunan Bileşen Sayısı: 1,229 / 3,111 (%39.5)


## 7. Doğrulama ve Teslim Kontrol Listesi

Kişi A sorumluluk alanındaki teslimatların doğrulanması:
- [x] SKRS ATC Adı sözlüğü doğru ve tutarlı oluşturuldu.
- [x] `ddinter_ilac.csv` yaklaşık 1.939 satır olarak çıkarıldı.
- [x] Yol A (tek bileşenli) ve Yol B (kombine) kurallara tam sadık kalınarak uygulandı.
- [x] Eşleşmeyen tüm bileşenler (1.123 adet) Kişi B'ye devredilmek üzere `yol='eslesmedi'` olarak işaretlendi.
- [x] `data/interim/etken_madde.csv` taslağı oluşturuldu ve Kişi B'ye (Yol C, Yol D ve PubChem eşleşmeleri için) devredilmeye hazır.


In [16]:
assert 1900 <= len(df_ddinter_ilac) <= 2000, f"HATA: ddinter_ilac satır sayısı (~1.939) aralık dışında: {len(df_ddinter_ilac)}!"
assert len(df_etken_madde) == 3111, f"HATA: etken_madde satır sayısı 3111 olmalı, {len(df_etken_madde)} çıktı!"
assert (df_etken_madde['yol'] == 'A').sum() == 1922, f"HATA: Yol A sayısı 1922 olmalı, {(df_etken_madde['yol'] == 'A').sum()} çıktı!"
assert (df_etken_madde['yol'] == 'B').sum() == 66, f"HATA: Yol B sayısı 66 olmalı, {(df_etken_madde['yol'] == 'B').sum()} çıktı!"
assert (df_etken_madde['yol'] == 'eslesmedi').sum() == 1123, f"HATA: Eşleşmedi sayısı 1123 olmalı, {(df_etken_madde['yol'] == 'eslesmedi').sum()} çıktı!"

print("════════════════════════════════════════════════════════════════")
print("✓ TÜM DOĞRULAMA KONTROLLERİ BAŞARIYLA GEÇTİ!")
print("✓ Kişi A İP3 teslimatı (Yol A ve Yol B) tamamlandı.")
print("✓ etken_madde.csv (Yol A/B) Kişi B'ye devredilmeye hazır.")
print("════════════════════════════════════════════════════════════════")


════════════════════════════════════════════════════════════════
✓ TÜM DOĞRULAMA KONTROLLERİ BAŞARIYLA GEÇTİ!
✓ Kişi A İP3 teslimatı (Yol A ve Yol B) tamamlandı.
✓ etken_madde.csv (Yol A/B) Kişi B'ye devredilmeye hazır.
════════════════════════════════════════════════════════════════


---
# ════════════════════════════════════════════════════════════════
# İP3 — Kişi B: Yol C, DDInter Eşleme, PubChem ve Raporlama (Gün 5–6)
# ════════════════════════════════════════════════════════════════

**Kişi B Sorumluluk Alanı ve İş Akışı:**
1. **Adım 3 — SKRS ve Yol C:** Kişi A'nın Yol A ve Yol B sonrasında bıraktığı bileşenlere SKRS (Kural 5/6) ve `config/tr_en_kurallar.csv` kural motorunu uygulama.
2. **Adım 4 — DDInter Eşlemesi:** DDInter `ad_norm` fihristi ile birebir eşleştirme, `ad_en_ham` izlenebilirliği ve `ddinter_id` atama.
3. **Adım 4 — PubChem En Sık 100 Kontrolü:** Kapsam içi en sık 100 eşleşmeyen bileşen için PubChem API ile kurtarma denemesi.
4. **Kişi C Taslağı:** Eşleşmeyen bileşenleri sıklık sırasına, `atc_ipucu` ve `urun_ornegi` bağlamıyla `data/interim/ruhsatli_esleme_taslagi.csv` olarak hazırlama.
5. **Teslimatlar:** `data/interim/etken_madde.csv` ve 4 zorunlu tabloyu içeren `docs/esleme_raporu.md` raporu.


In [18]:
# ─── 8. Yol C: Kural Tabanlı Eşleme Motoru (tr_en_kurallar.csv) ────────────
import re

def suffix_specificity(pattern: str) -> int:
    """Sonek kuralının özgüllüğünü hesaplar (harf sayısı)."""
    return len(re.sub(r'[^a-zA-Z]', '', str(pattern)))

def kural_adaylari(bilesen_tr: str, kurallar_df: pd.DataFrame) -> list:
    """Türkçe etken madde adı için kurallardan öncelik sıralı İngilizce aday listesi üretir."""
    if not isinstance(bilesen_tr, str) or not bilesen_tr.strip():
        return []
    ham = bilesen_tr.lower().strip()
    df_rules = kurallar_df.dropna(subset=['kural_tipi', 'eski', 'yeni']).copy()
    suffix_rules = df_rules[df_rules['kural_tipi'] == 'suffix_replace'].copy()
    suffix_rules['_spec'] = suffix_rules['eski'].apply(suffix_specificity)
    suffix_rules = suffix_rules.sort_values('_spec', ascending=False)
    char_rules = df_rules[df_rules['kural_tipi'] == 'char_replace']

    # 1. Sonek kurallarına uyan adaylar
    sonek_adaylar = []
    for _, kural in suffix_rules.iterrows():
        eski, yeni = str(kural['eski']), str(kural['yeni'])
        if re.search(eski, ham):
            sonek_adaylar.append(re.sub(eski, yeni, ham))

    base_forms = sonek_adaylar + [ham]
    adaylar = []

    # 2. Tüm karakter kuralları birlikte uygulandığında
    for base in base_forms:
        all_char = base
        for _, kural in char_rules.iterrows():
            all_char = all_char.replace(str(kural['eski']), str(kural['yeni']))
        if all_char not in adaylar:
            adaylar.append(all_char)

    # 3. Tek tek karakter kuralları uygulandığında
    for base in base_forms:
        for _, kural in char_rules.iterrows():
            c_cand = base.replace(str(kural['eski']), str(kural['yeni']))
            if c_cand not in adaylar:
                adaylar.append(c_cand)

    # 4. Doğrudan baz formlar (sadece sonek veya ham)
    for base in base_forms:
        if base not in adaylar:
            adaylar.append(base)

    return adaylar

def yol_c_esle(bilesen_tr: str, ddinter_norm_set: set, kurallar_df: pd.DataFrame):
    """Kural adaylarından DDInter'da bulunan ilk ad_norm değerini döner, yoksa None döner."""
    for aday in kural_adaylari(bilesen_tr, kurallar_df):
        norm = ad_normalize(aday)
        if norm in ddinter_norm_set:
            return norm
    return None

def yol_a_olcum(yol_a_df: pd.DataFrame, kurallar_df: pd.DataFrame):
    """Kuralların Yol A test çiftleri (1.922 çift) üzerindeki tahmin başarısını ölçer."""
    df = yol_a_df[['bilesen_tr', 'ad_en']].dropna().copy()
    ok = df.apply(
        lambda r: ad_normalize(r['ad_en']) in [
            ad_normalize(a) for a in kural_adaylari(r['bilesen_tr'], kurallar_df)
        ],
        axis=1
    )
    basari_orani = float(ok.mean() * 100)
    tutmayanlar = df[~ok].copy()
    return basari_orani, tutmayanlar

kurallar_yol = kok / 'config' / 'tr_en_kurallar.csv'
df_kurallar = pd.read_csv(kurallar_yol)
ddinter_norm_set = set(df_ddinter_ilac['ad_norm'].dropna().tolist())

# Yol A test seti (1.922 çift) üzerinde kural başarı ölçümü
basari_orani, _ = yol_a_olcum(df_etken_madde[df_etken_madde['yol'] == 'A'], df_kurallar)

print(f"✓ Yol C kural motoru yüklendi ({len(df_kurallar)} kural).")
print(f"✓ Yol A test çiftleri üzerindeki kural başarı oranı: %{basari_orani:.1f}")
print()
print("Örnek Kural Testleri:")
for test_tr in ["amlodipin", "gentamisin", "morfin", "parasetamol", "fluoksetin", "levodopa", "omeprazol", "diklofenak", "indapamid", "budesonid"]:
    eslesti = yol_c_esle(test_tr, ddinter_norm_set, df_kurallar)
    durum = f"DDInter Eşleşti: {eslesti}" if eslesti else "DDInter'da Yok"
    print(f"  {test_tr:<15} -> {durum}")

# Regresyon Koruma Testi (Amlodipin, Gentamisin, Morfin, Fluoksetin)
for tr, en in [('amlodipin', 'amlodipine'), ('gentamisin', 'gentamicin'), ('morfin', 'morphine'), ('fluoksetin', 'fluoxetine')]:
    assert yol_c_esle(tr, ddinter_norm_set, df_kurallar) == en, f'Yol C regresyon: {tr}'
print("✓ Tüm Yol C regresyon testleri başarıyla doğrulandı.")


✓ Yol C kural motoru yüklendi (9 kural).
✓ Yol A test çiftleri üzerindeki kural başarı oranı: %52.7

Örnek Kural Testleri:
  amlodipin       -> DDInter Eşleşti: amlodipine
  gentamisin      -> DDInter Eşleşti: gentamicin
  morfin          -> DDInter Eşleşti: morphine
  parasetamol     -> DDInter'da Yok
  fluoksetin      -> DDInter Eşleşti: fluoxetine
  levodopa        -> DDInter Eşleşti: levodopa
  omeprazol       -> DDInter Eşleşti: omeprazole
  diklofenak      -> DDInter Eşleşti: diclofenac
  indapamid       -> DDInter Eşleşti: indapamide
  budesonid       -> DDInter Eşleşti: budesonide
✓ Tüm Yol C regresyon testleri başarıyla doğrulandı.


In [19]:
# ─── 9. SKRS (Kural 5/6) ve Yol C Uygulanması ────────────────────────────
# 1. Kapsam İçi / Kapsam Dışı Bayrağının Hesaplanması (Kural 11)
kd = df_ue.merge(df_urun[['barkod', 'kapsam_disi']], on='barkod')
kd['kapsam_disi'] = pd.to_numeric(kd['kapsam_disi'], errors='coerce').fillna(0).astype(int)
kapsam_ici_kumesi = set(kd.loc[kd['kapsam_disi'] == 0, 'bilesen_tr'])
df_etken_madde['kapsam_ici'] = df_etken_madde['bilesen_tr'].isin(kapsam_ici_kumesi).astype(int)

# 2. SKRS Ürünlerinin Bileşenleri (Sabit Kural 5 ve Kural 6):
# SKRS'te aktif olup ruhsatlıda olmayan ürünlerin bileşenleri zaten İngilizcedir (ATC adından bölünmüştür).
# Bu nedenle Türkçe çeviri motoruna sokulmadan doğrudan İngilizce ad olarak atanır ve yol='SKRS' olur.
# İngilizce adda Türkçe noktasız 'ı' hatası temizlenir (örn: combınatıon -> combination)
skrs_bar = set(df_urun.loc[df_urun['kaynak'] == 'SKRS', 'barkod'])
skrs_bilesenler = set(df_ue[df_ue['barkod'].isin(skrs_bar)]['bilesen_tr'])
mask_skrs = (df_etken_madde['yol'] == 'eslesmedi') & (df_etken_madde['bilesen_tr'].isin(skrs_bilesenler))
df_etken_madde.loc[mask_skrs, 'ad_en'] = (
    df_etken_madde.loc[mask_skrs, 'bilesen_tr']
    .str.replace('ı', 'i', regex=False)
    .apply(ad_normalize)
)
df_etken_madde.loc[mask_skrs, 'yol'] = 'SKRS'
skrs_sayisi = int(mask_skrs.sum())

# 3. Yol C: Kalan Eşleşmemiş Bileşenlere Çoklu Adaylı Kural Motoru
mask_eslesmedi = df_etken_madde['yol'] == 'eslesmedi'
yol_c_adaylar = df_etken_madde.loc[mask_eslesmedi, 'bilesen_tr'].apply(
    lambda b: yol_c_esle(b, ddinter_norm_set, df_kurallar)
)
basarili = yol_c_adaylar.dropna()
df_etken_madde.loc[basarili.index, 'ad_en'] = basarili
df_etken_madde.loc[basarili.index, 'yol'] = 'C'
yol_c_sayisi = len(basarili)

print(f"✓ SKRS adımı tamamlandı: {skrs_sayisi} bileşen (yol='SKRS')")
print(f"✓ Yol C eşleştirmesi tamamlandı: {yol_c_sayisi} yeni bileşen DDInter'a bağlandı (yol='C')")
print()
print("Yol Dağılımı (SKRS ve Yol C Sonrası):")
print(df_etken_madde['yol'].value_counts())


✓ SKRS adımı tamamlandı: 90 bileşen (yol='SKRS')
✓ Yol C eşleştirmesi tamamlandı: 78 yeni bileşen DDInter'a bağlandı (yol='C')

Yol Dağılımı (SKRS ve Yol C Sonrası):
yol
A            1922
eslesmedi     955
SKRS           90
C              78
B              66
Name: count, dtype: int64


In [20]:
# ─── 10. DDInter ID Ataması (`ddinter_id`) ve İzlenebilirlik (`ad_en_ham`) ──
df_etken_madde['ad_en_ham'] = df_etken_madde['ad_en'].copy()  # Yol A/B/C/SKRS orijinal adı korunur
df_etken_madde['ad_norm'] = df_etken_madde['ad_en'].apply(
    lambda x: ad_normalize(str(x)) if pd.notna(x) and str(x).strip() else None
)
df_etken_madde['ddinter_id'] = df_etken_madde['ad_norm'].map(ddinter_dict)
df_etken_madde['eslesme_kaynagi'] = np.where(df_etken_madde['ddinter_id'].notna(), 'tam_ad', None)

print("✓ DDInter ID eşleştirmesi tamamlandı:")
for y in ['A', 'B', 'C', 'SKRS', 'eslesmedi']:
    sub = df_etken_madde[df_etken_madde['yol'] == y]
    matched = sub['ddinter_id'].notna().sum()
    toplam = len(sub)
    oran = (matched / toplam * 100) if toplam > 0 else 0
    print(f"  Yol {y:<9}: Toplam = {toplam:>4}, DDInter Eşleşen = {matched:>4} (%{oran:5.1f})")

toplam_eslesen = df_etken_madde['ddinter_id'].notna().sum()
print()
print(f"Toplam DDInter ile Eşleşen Etken Madde: {toplam_eslesen:,} / {len(df_etken_madde):,} (%{toplam_eslesen/len(df_etken_madde)*100:.2f})")


✓ DDInter ID eşleştirmesi tamamlandı:
  Yol A        : Toplam = 1922, DDInter Eşleşen = 1197 (% 62.3)
  Yol B        : Toplam =   66, DDInter Eşleşen =   32 (% 48.5)
  Yol C        : Toplam =   78, DDInter Eşleşen =   78 (%100.0)
  Yol SKRS     : Toplam =   90, DDInter Eşleşen =   31 (% 34.4)
  Yol eslesmedi: Toplam =  955, DDInter Eşleşen =    0 (%  0.0)

Toplam DDInter ile Eşleşen Etken Madde: 1,338 / 3,111 (%43.01)


In [21]:
# ─── 11. PubChem Eş Anlamlı Kontrolü (En Sık 100 Kapsam İçi Eşleşmemiş) ───
import hashlib
import json
import time
import urllib.parse
import requests

CACHE_DIR = kok / 'data' / 'cache' / 'pubchem'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LAST_PUBCHEM_TIME = 0.0
PUBCHEM_HEADERS = {'User-Agent': 'DDInter_Pipeline/1.0 (academic-research; ddi-project)'}

def pubchem_synonyms_guvenli(ad: str, cache_dir: Path = CACHE_DIR, max_retries: int = 3) -> list:
    """PubChem REST API'den güvenle eş anlamlı listesi çeker.
    
    Özellikler:
    - Rate Limit: İstekler arası >= 0.20 sn bekler (en fazla 5 istek/sn).
    - Cache: data/cache/pubchem/<md5>.json altında saklar.
    - 404 Koruması: Bulunamayan bileşikler boş liste [] olarak önbelleğe yazılır, mükerrer ağ istekleri önlenir.
    """
    global LAST_PUBCHEM_TIME
    if not ad or pd.isna(ad):
        return []
    clean = ad_normalize(str(ad))
    if not clean:
        return []
    h = hashlib.md5(clean.encode('utf-8')).hexdigest()
    f = cache_dir / f"{h}.json"
    if f.exists():
        try:
            if f.stat().st_size > 0:
                with open(f, 'r', encoding='utf-8') as fp:
                    d = json.load(fp)
                    return d if isinstance(d, list) else d.get('synonyms', [])
        except Exception:
            pass
    encoded_name = urllib.parse.quote(clean)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{encoded_name}/synonyms/JSON"
    synonyms = []
    should_cache = False
    for attempt in range(1, max_retries + 1):
        elapsed = time.time() - LAST_PUBCHEM_TIME
        if elapsed < 0.20:
            time.sleep(0.20 - elapsed)
        LAST_PUBCHEM_TIME = time.time()
        try:
            resp = requests.get(url, headers=PUBCHEM_HEADERS, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                info = data.get('InformationList', {}).get('Information', [])
                if info:
                    synonyms = info[0].get('Synonym', [])
                should_cache = True
                break
            elif resp.status_code == 404:
                synonyms = []
                should_cache = True
                break
            elif resp.status_code in (429, 500, 502, 503, 504):
                time.sleep(attempt * 1.0)
                continue
            else:
                return []
        except requests.RequestException:
            time.sleep(attempt * 1.0)
            continue
    if should_cache:
        try:
            f.write_text(json.dumps(synonyms, ensure_ascii=False), encoding='utf-8')
        except Exception:
            pass
    return synonyms

# DDInter'da eşleşmeyen en sık 100 kapsam içi bileşen
df_eslesmeyen_top100 = (
    df_etken_madde[df_etken_madde['ddinter_id'].isna() & (df_etken_madde['kapsam_ici'] == 1)]
    .sort_values(by=['urun_sayisi', 'bilesen_tr'], ascending=[False, True])
    .head(100).copy()
)

pubchem_kurtarilan = 0
kurtarilan_detaylar = []
for row in df_eslesmeyen_top100.itertuples():
    idx = row.Index
    aday_adlar = []
    if pd.notna(row.ad_en) and str(row.ad_en).strip():
        aday_adlar.append(str(row.ad_en))
    if pd.notna(row.bilesen_tr) and str(row.bilesen_tr).strip():
        aday_adlar.append(str(row.bilesen_tr))
        
    eslesti = False
    for ad in aday_adlar:
        if eslesti:
            break
        syns = pubchem_synonyms_guvenli(ad, cache_dir=CACHE_DIR)
        for syn in syns:
            norm_syn = ad_normalize(syn)
            if norm_syn in ddinter_dict:
                matched_id = ddinter_dict[norm_syn]
                df_etken_madde.at[idx, 'ddinter_id'] = matched_id
                df_etken_madde.at[idx, 'ad_norm'] = norm_syn
                df_etken_madde.at[idx, 'ad_en'] = norm_syn
                df_etken_madde.at[idx, 'eslesme_kaynagi'] = 'pubchem'
                pubchem_kurtarilan += 1
                kurtarilan_detaylar.append((row.bilesen_tr, norm_syn, matched_id))
                eslesti = True
                break

# Kurtarılan PubChem eşleşmelerinin aynı ham İngilizce ada sahip diğer satırlara yayılması
# (Örn: parasetamol -> acetaminophen kurtarılınca, SKRS'ten gelen paracetamol satırı da acetaminophen'e bağlanır)
recovered_syn_map = {}
for idx in df_etken_madde[df_etken_madde['eslesme_kaynagi'] == 'pubchem'].index:
    norm_ham = ad_normalize(str(df_etken_madde.at[idx, 'ad_en_ham'])) if pd.notna(df_etken_madde.at[idx, 'ad_en_ham']) else ''
    norm_tr = ad_normalize(str(df_etken_madde.at[idx, 'bilesen_tr']))
    target_syn = df_etken_madde.at[idx, 'ad_en']
    target_id = df_etken_madde.at[idx, 'ddinter_id']
    if norm_ham:
        recovered_syn_map[norm_ham] = (target_syn, target_id)
    if norm_tr:
        recovered_syn_map[norm_tr] = (target_syn, target_id)

yayilim_sayisi = 0
for idx in df_etken_madde[df_etken_madde['ddinter_id'].isna()].index:
    norm_ham = ad_normalize(str(df_etken_madde.at[idx, 'ad_en_ham'])) if pd.notna(df_etken_madde.at[idx, 'ad_en_ham']) else ''
    norm_tr = ad_normalize(str(df_etken_madde.at[idx, 'bilesen_tr']))
    matched = None
    if norm_ham and norm_ham in recovered_syn_map:
        matched = recovered_syn_map[norm_ham]
    elif norm_tr and norm_tr in recovered_syn_map:
        matched = recovered_syn_map[norm_tr]
    if matched:
        df_etken_madde.at[idx, 'ddinter_id'] = matched[1]
        df_etken_madde.at[idx, 'ad_norm'] = matched[0]
        df_etken_madde.at[idx, 'ad_en'] = matched[0]
        df_etken_madde.at[idx, 'eslesme_kaynagi'] = 'pubchem_yayilim'
        yayilim_sayisi += 1

print(f"✓ PubChem taraması tamamlandı: En sık 100 kapsam içi eşleşmeyenden {pubchem_kurtarilan} ek etken madde DDInter'a bağlandı.")
if yayilim_sayisi > 0:
    print(f"✓ Kurtarılan eşleşmeler aynı ham ada sahip {yayilim_sayisi} ek bileşene uygulandı (örn. SKRS paracetamol -> acetaminophen).")
print("Gözle Kontrol (Kurtarılan Eşleşmeler):")
for b_tr, syn_ad, did in kurtarilan_detaylar:
    print(f"  {b_tr:<20} -> {syn_ad:<22} ({did})")


✓ PubChem taraması tamamlandı: En sık 100 kapsam içi eşleşmeyenden 16 ek etken madde DDInter'a bağlandı.
✓ Kurtarılan eşleşmeler aynı ham ada sahip 24 ek bileşene uygulandı (örn. SKRS paracetamol -> acetaminophen).
Gözle Kontrol (Kurtarılan Eşleşmeler):
  parasetamol          -> acetaminophen          (DDInter14)
  olmesartan           -> olmesartan             (DDInter1330)
  vitamin d3           -> cholecalciferol        (DDInter367)
  kolekalsiferol       -> cholecalciferol        (DDInter367)
  klorfeniramin        -> chlorpheniramine       (DDInter362)
  askorbik asit        -> ascorbic acid          (DDInter124)
  alfa lipoik asit     -> lipoic acid            (DDInter1076)
  tiotropium           -> tiotropium             (DDInter1815)
  piridoksin           -> pyridoxine             (DDInter1549)
  siklosporin          -> cyclosporine           (DDInter458)
  asiklovir            -> acyclovir              (DDInter25)
  glibenklamid         -> glyburide              (DDInter831)


In [22]:
# ─── 12. Kişi C İçin Yol D Taslağı (ruhsatli_esleme_taslagi.csv) ───────────
# DDInter ile eşleşmeyen tüm bileşenler; kapsam_ici=1 olanlar öne alınarak sıralanır.
# Kişi C'nin elle eşlemesini kolaylaştırmak için 'atc_ipucu' ve 'urun_ornegi' eklenir.
taslak_c = df_etken_madde[df_etken_madde['ddinter_id'].isna()].copy()
taslak_c['durum'] = np.where(
    taslak_c['ad_en'].notna() & (taslak_c['ad_en'].astype(str).str.strip() != ''),
    'ingilizce_var_ddinter_yok',
    'ceviri_yok'
)

_ua = df_ue.merge(df_urun[['barkod', 'atc_final', 'urun_adi']], on='barkod')
atc_ornek = _ua.dropna(subset=['atc_final']).groupby('bilesen_tr')['atc_final'].agg(
    lambda s: s.mode().iloc[0] if not s.empty and not s.mode().empty else None
)
urun_ornek = _ua.groupby('bilesen_tr')['urun_adi'].first()
taslak_c['atc_ipucu'] = taslak_c['bilesen_tr'].map(atc_ornek)
taslak_c['urun_ornegi'] = taslak_c['bilesen_tr'].map(urun_ornek)
taslak_c = taslak_c.sort_values(
    by=['kapsam_ici', 'urun_sayisi', 'bilesen_tr'],
    ascending=[False, False, True]
).reset_index(drop=True)

taslak_c_yol = kok / 'data' / 'interim' / 'ruhsatli_esleme_taslagi.csv'
sutunlar_taslak = ['bilesen_tr', 'ad_en', 'atc', 'atc_ipucu', 'yol', 'urun_sayisi', 'kapsam_ici', 'durum', 'urun_ornegi']
taslak_c[sutunlar_taslak].to_csv(taslak_c_yol, index=False)

print(f"✓ Kişi C için ruhsatli_esleme_taslagi.csv kaydedildi: {taslak_c_yol}")
print(f"  Eşleşmeyen toplam bileşen sayısı: {len(taslak_c):,}")
print(f"  Bunlardan kapsam içi olanlar     : {(taslak_c['kapsam_ici'] == 1).sum():,}")
print()
print("En Sık Eşleşmeyen İlk 10 Kapsam İçi Bileşen (Kişi C Öncelikli & İpuçlu):")
print(taslak_c[['bilesen_tr', 'urun_sayisi', 'atc_ipucu', 'durum', 'urun_ornegi']].head(10))


✓ Kişi C için ruhsatli_esleme_taslagi.csv kaydedildi: /Users/reyhanbuseyanar/Desktop/ddi-final/data/interim/ruhsatli_esleme_taslagi.csv
  Eşleşmeyen toplam bileşen sayısı: 1,733
  Bunlardan kapsam içi olanlar     : 1,610

En Sık Eşleşmeyen İlk 10 Kapsam İçi Bileşen (Kişi C Öncelikli & İpuçlu):
          bilesen_tr  urun_sayisi atc_ipucu                      durum                                        urun_ornegi
0           dekstroz          720   B05BB02  ingilizce_var_ddinter_yok  İE % 5 DEKSTROZ İZOTONİK SODYUM KLORÜR SOLÜSYO...
1    kalsiyum klorur          400      B05D                 ceviri_yok  Primefleks Ringer 0,86 g+0,03 g+0,033 g/100 mL...
2  hidroklorotiyazid          379   C09DA03                 ceviri_yok  EXCALİBA PLUS 40 MG/10 MG/25 MG FİLM KAPLI TAB...
3      sodyum laktat          315      B05D                 ceviri_yok  Biyonefrol 40 - % 1.36 Glukoz İçeren Periton D...
4   magnezyum klorur          293      B05D                 ceviri_yok  Primefleks İzomax-S pH 

In [23]:
# ─── 13. Nihai etken_madde.csv Tablosunun Kaydedilmesi ────────────────────
# ATC Ana Grubu Hesaplanması (Mode esasına göre ilk harf A-V; D05 ve E09 için zorunlu)
ue_atc = df_ue.merge(df_urun[['barkod', 'atc_final']], on='barkod').dropna(subset=['atc_final'])
ue_atc = ue_atc[ue_atc['atc_final'].astype(str).str.len() >= 1]
atc_grup = ue_atc.groupby('bilesen_tr')['atc_final'].agg(
    lambda s: s.astype(str).str[:1].str.upper().mode().iloc[0] if not s.empty and not s.mode().empty else None
)
df_etken_madde['atc_ana'] = df_etken_madde['bilesen_tr'].map(atc_grup)

# Plandaki standart sütunlar ve ek izleme sütunları (ad_en_ham, atc_ana, kapsam_ici, eslesme_kaynagi)
sutunlar = ['bilesen_tr', 'ad_en', 'ad_en_ham', 'atc', 'atc_ana', 'yol', 'urun_sayisi', 'ddinter_id', 'kapsam_ici', 'eslesme_kaynagi']
df_etken_madde_nihai = df_etken_madde[sutunlar].sort_values(
    by=['urun_sayisi', 'bilesen_tr'], ascending=[False, True]
).reset_index(drop=True)

etken_cikti_yol = kok / 'data' / 'interim' / 'etken_madde.csv'
df_etken_madde_nihai.to_csv(etken_cikti_yol, index=False)

print(f"✓ Nihai etken_madde.csv başarıyla kaydedildi: {etken_cikti_yol}")
print(f"  Toplam satır sayısı          : {len(df_etken_madde_nihai):,}")
print(f"  DDInter ID içeren satır sayısı: {df_etken_madde_nihai['ddinter_id'].notna().sum():,}")
print(f"  Kapsam içi satır sayısı       : {(df_etken_madde_nihai['kapsam_ici'] == 1).sum():,}")


✓ Nihai etken_madde.csv başarıyla kaydedildi: /Users/reyhanbuseyanar/Desktop/ddi-final/data/interim/etken_madde.csv
  Toplam satır sayısı          : 3,111
  DDInter ID içeren satır sayısı: 1,378
  Kapsam içi satır sayısı       : 2,988


In [24]:
# ─── 14. Eşleşme Raporu ve 4 Zorunlu Tablo (docs/esleme_raporu.md) ─────────
total_etken = len(df_etken_madde_nihai)
matched_etken = int(df_etken_madde_nihai['ddinter_id'].notna().sum())
matched_etken_pct = (matched_etken / total_etken * 100) if total_etken > 0 else 0

# Kapsam içi etken madde metrikleri
kapsam_ici_etken = df_etken_madde_nihai[df_etken_madde_nihai['kapsam_ici'] == 1]
total_kapsam_ici_etken = len(kapsam_ici_etken)
matched_kapsam_ici_etken = int(kapsam_ici_etken['ddinter_id'].notna().sum())
matched_kapsam_ici_etken_pct = (matched_kapsam_ici_etken / total_kapsam_ici_etken * 100) if total_kapsam_ici_etken > 0 else 0

# 1. Ürün Kapsama Oranı (Ruhsatlı ürünler) - Doğrudan bilesen_tr ile Temiz Birleştirme
ruhsatli_barkodlar = set(df_urun.loc[df_urun['kaynak'] == 'Ruhsatlı', 'barkod'])
df_ue_ruhsatli = df_ue[df_ue['barkod'].isin(ruhsatli_barkodlar)].copy()

ddinter_bilesen_kumesi = set(df_etken_madde_nihai.loc[df_etken_madde_nihai['ddinter_id'].notna(), 'bilesen_tr'])
kapsanan_barkodlar = set(df_ue_ruhsatli[df_ue_ruhsatli['bilesen_tr'].isin(ddinter_bilesen_kumesi)]['barkod'])

total_urun = len(ruhsatli_barkodlar)
matched_urun = len(kapsanan_barkodlar)
matched_urun_pct = (matched_urun / total_urun * 100) if total_urun > 0 else 0

# Kapsam içi ürün metrikleri (kapsam_disi == 0)
kapsam_ici_barkodlar = set(df_urun.loc[(df_urun['kaynak'] == 'Ruhsatlı') & (df_urun['kapsam_disi'] == 0), 'barkod'])
kapsanan_kapsam_ici_barkodlar = kapsanan_barkodlar & kapsam_ici_barkodlar
total_kapsam_ici_urun = len(kapsam_ici_barkodlar)
matched_kapsam_ici_urun = len(kapsanan_kapsam_ici_barkodlar)
matched_kapsam_ici_urun_pct = (matched_kapsam_ici_urun / total_kapsam_ici_urun * 100) if total_kapsam_ici_urun > 0 else 0

# Tablo 1: Yol Bazında Bileşen Sayısı ve DDInter Eşleşme Oranı
yol_tanimlari = [
    ('A', 'Yol A (Tek Bileşenli ATC)', 'Tek bileşenli ürünlerin SKRS ATC sözlüğünden İngilizce adı'),
    ('B', 'Yol B (Kombine Bölme & RapidFuzz)', 'Kombine ürünlerin ATC adından ayrıştırılan bileşenler'),
    ('C', 'Yol C (Kural Tabanlı - tr_en_kurallar)', 'Türkçe kural tabanlı çeviri motoru ile eşleşenler'),
    ('SKRS', 'SKRS (Kural 5/6)', 'SKRS\'ten doğrudan İngilizce gelen ürünler'),
    ('eslesmedi', 'Eşleşmedi (Kişi C - Yol D Adayları)', 'Kural motoru ve PubChem ile eşleşmeyip Kişi C uzman incelemesine kalanlar'),
]
tablo_1_satirlar = []
for y_kod, y_ad, aciklama in yol_tanimlari:
    sub = df_etken_madde_nihai[df_etken_madde_nihai['yol'] == y_kod]
    tot = len(sub)
    esl = int(sub['ddinter_id'].notna().sum())
    oran = (esl / tot * 100) if tot > 0 else 0
    tablo_1_satirlar.append((y_ad, tot, esl, oran, aciklama))

# Tablo 3: ATC Ana Grubuna Göre DDInter Eşleşme Oranı (14 Satır - Kapsam İçi Filtreli)
valid_atc_em = df_etken_madde_nihai[
    (df_etken_madde_nihai['kapsam_ici'] == 1) & df_etken_madde_nihai['atc_ana'].isin(list('ABCDGHJLMNPRSV'))
]

atc_gruplari = []
for g in sorted(list('ABCDGHJLMNPRSV')):
    sub = valid_atc_em[valid_atc_em['atc_ana'] == g]
    tot = len(sub)
    esl = int(sub['ddinter_id'].notna().sum())
    oran = (esl / tot * 100) if tot > 0 else 0
    atc_gruplari.append((g, tot, esl, oran))

# Tablo 4: Eşleşmeyen En Sık 30 Bileşen (Kapsam İçi Filtreli Öncelik Listesi)
eslesmeyen_top30 = df_etken_madde_nihai[
    df_etken_madde_nihai['ddinter_id'].isna() & (df_etken_madde_nihai['kapsam_ici'] == 1)
].head(30)

# Parasetamol ve PubChem metrikleri (dinamik ve güvenli)
pubchem_dogrudan = int((df_etken_madde_nihai['eslesme_kaynagi'] == 'pubchem').sum())
pubchem_yayilim_cnt = int((df_etken_madde_nihai['eslesme_kaynagi'] == 'pubchem_yayilim').sum())
parasetamol_info = ""
p = df_etken_madde_nihai[
    (df_etken_madde_nihai['bilesen_tr'] == 'parasetamol')
    & (df_etken_madde_nihai['eslesme_kaynagi'].isin(['pubchem', 'pubchem_yayilim']))
]
if not p.empty:
    r = p.iloc[0]
    parasetamol_info = f" Örneğin {int(r['urun_sayisi'])} ürünlü parasetamol → {r['ddinter_id']} ({r['ad_en']})."

# Markdown Rapor Metni Oluştur
rapor_lines = [
    "# İP3: DDInter İlaç Eşleşme Raporu",
    "",
    f"> **Özet:** Türkiye'deki kapsam içi etken maddelerin %{matched_kapsam_ici_etken_pct:.1f}'i ({matched_kapsam_ici_etken:,} / {total_kapsam_ici_etken:,}), kapsam içi ruhsatlı ürünlerin %{matched_kapsam_ici_urun_pct:.1f}'inin ({matched_kapsam_ici_urun:,} / {total_kapsam_ici_urun:,}) en az bir bileşeni DDInter'da bulunuyor.",
    f"> *(Tüm Bileşenler Bazında: %{matched_etken_pct:.1f} ({matched_etken:,} / {total_etken:,}) — Tüm Ruhsatlı Ürünler Bazında: %{matched_urun_pct:.1f} ({matched_urun:,} / {total_urun:,}))*",
    "",
    "## 1. Yol Bazında Bileşen Sayısı ve DDInter Eşleşme Dağılımı",
    "| Eşleşme Yolu | Toplam Bileşen | DDInter Eşleşen | Eşleşme Oranı (%) | Açıklama |",
    "| :--- | :---: | :---: | :---: | :--- |",
]
for yol_adi, tot, esl, pct, aciklama in tablo_1_satirlar:
    rapor_lines.append(f"| **{yol_adi}** | {tot:,} | {esl:,} | %{pct:.1f} | {aciklama} |")
rapor_lines.append(f"| **TOPLAM** | **{total_etken:,}** | **{matched_etken:,}** | **%{matched_etken_pct:.1f}** | TİTCK ruhsatlı listedeki tüm benzersiz bileşenler |")
rapor_lines.append("")
rapor_lines.append(f"> **Not (PubChem Doğrulaması ve Dağılımı):** Adım 4 kapsamında en sık 100 eşleşmeyen kapsam içi bileşenden {pubchem_dogrudan} tanesi PubChem eş anlamlılarıyla doğrudan DDInter kimliğine bağlanmış ve {pubchem_yayilim_cnt} satıra yayılmıştır.{parasetamol_info} Kurtarılan bu bileşikler resmi ATC adı ile DDInter adı farklılaşan Yol A ve Yol B satırlarıdır (bu nedenle Yol A/B altında DDInter kimliği kazanmışlardır). Kural motoruna girmeyip `eslesmedi` yolunda kalan 955 bileşenin en sık olanları (dekstroz, kalsiyum klorur, sodyum laktat, magnezyum klorur, tiyokolsikosid vb.) DDInter kütüphanesinde hiç yer almadığından bu gruptan yeni DDInter eşleşmesi çıkmamış ve tümü Kişi C'nin Yol D taslağına devredilmiştir.")

rapor_lines.extend([
    "",
    "## 2. DDInter Eşleşme Oranı Özeti",
    "| Metrik Düzeyi | Toplam Büyüklük | DDInter'da Bulunan | Kapsama Oranı (%) |",
    "| :--- | :---: | :---: | :--- |",
    f"| **Kapsam İçi Etken Madde Bazında** | {total_kapsam_ici_etken:,} | {matched_kapsam_ici_etken:,} | %{matched_kapsam_ici_etken_pct:.1f} |",
    f"| **Kapsam İçi Ruhsatlı Ürün Bazında (En Az 1 Bileşen)** | {total_kapsam_ici_urun:,} | {matched_kapsam_ici_urun:,} | %{matched_kapsam_ici_urun_pct:.1f} |",
    f"| **Tüm Etken Maddeler Bazında** | {total_etken:,} | {matched_etken:,} | %{matched_etken_pct:.1f} |",
    f"| **Tüm Ruhsatlı Ürünler Bazında (En Az 1 Bileşen)** | {total_urun:,} | {matched_urun:,} | %{matched_urun_pct:.1f} |",
    "",
    f"> **Not (Benzersiz İlaç Kimliği):** DDInter ile eşleşen {matched_etken:,} etken madde kaydı, toplam **{int(df_etken_madde_nihai['ddinter_id'].nunique()):,} benzersiz DDInter kimliğine** (`ddinter_id`) karşılık gelmektedir (örn. `vitamin d3` ve `kolekalsiferol` her ikisi de `DDInter367` kimliğine bağlanmıştır).",
    "",
    "## 3. ATC Ana Grubuna Göre DDInter Eşleşme Oranı (14 Satır - Kapsam İçi)",
    "| ATC Ana Grubu | Grup Açıklaması | Toplam Bileşen | DDInter Eşleşen | Eşleşme Oranı (%) |",
    "| :---: | :--- | :---: | :--- | :---: |",
])

atc_isimleri = {
    'A': 'Sindirim Sistemi ve Metabolizma',
    'B': 'Kan ve Kan Yapıcı Organlar',
    'C': 'Kardiyovasküler Sistem',
    'D': 'Dermatolojik İlaçlar',
    'G': 'Genito-Üriner Sistem ve Cinsiyet Hormonları',
    'H': 'Sistemik Hormonal Preparatlar',
    'J': 'Sistemik Antiinfektifler (Antibiyotikler vb.)',
    'L': 'Antineoplastik ve İmmünomodülatör Ajanlar',
    'M': 'Kas-İskelet Sistemi',
    'N': 'Sinir Sistemi',
    'P': 'Antiparaziter Ürünler',
    'R': 'Solunum Sistemi',
    'S': 'Duyu Organları',
    'V': 'Çeşitli (Various)',
}

for g, tot, esl, oran in atc_gruplari:
    g_ad = atc_isimleri.get(g, '-')
    rapor_lines.append(f"| **{g}** | {g_ad} | {tot:,} | {esl:,} | %{oran:.1f} |")

atc_disinda_kalan = total_etken - len(valid_atc_em)
rapor_lines.append("")
rapor_lines.append(f"> **Not (ATC Dağılımı):** Tablo kapsam içi {len(valid_atc_em):,} etken maddeyi içerir. Kapsam dışı veya ATC atanmamış {atc_disinda_kalan:,} bileşen tablodan hariç tutulmuştur; toplam etken madde {total_etken:,}'dir.")
rapor_lines.append("")
n_oran = next((oran for g, _, _, oran in atc_gruplari if g == 'N'), 69.8)
rapor_lines.append(f"> **Önemli Not (Eşleşme Oranı vs. Etkileşim Kapsamı):** Tablodaki yüksek eşleşme oranları (örn. Sinir Sistemi %{n_oran:.1f}), bu ilaçların DDInter kütüphanesinde kimlik olarak geçtiğini gösterir. Ancak DDInter indirme paketlerinde C, G, J, M, N, S gibi ana grupların bağımsız etkileşim dosyaları bulunmamaktadır. Dolayısıyla bu gruptaki ilaçlar yalnızca diğer mevcut gruplarla (A, B, D, H, L, P, R, V) olan etkileşimlerde yer alabilir; iki N grubu ilacının kendi arasındaki etkileşim DDInter dosyalarında bulunamaz.")

rapor_lines.extend([
    "",
    "## 4. Eşleşmeyen En Sık 30 Bileşen Tablosu (Kişi C Öncelik Listesi - Kapsam İçi)",
    "| Sıra | Türkçe Bileşen Adı | Ürün Sayısı | Önerilen İngilizce / ATC | Durum |",
    "| :---: | :--- | :---: | :--- | :--- |",
])

for rank, (_, row) in enumerate(eslesmeyen_top30.iterrows(), 1):
    b_tr = row['bilesen_tr']
    u_cnt = row['urun_sayisi']
    atc_val = row['atc'] if pd.notna(row['atc']) else '-'
    ad_val = row['ad_en'] if pd.notna(row['ad_en']) else '-'
    rapor_lines.append(f"| {rank} | `{b_tr}` | {u_cnt:,} | {ad_val} ({atc_val}) | Yol D Adayı |")

# Bütünlük Doğrulama Kontrolleri (Assertions - Dosya yazılmadan önce doğrulanır)
assert df_etken_madde_nihai['bilesen_tr'].is_unique, 'HATA: bilesen_tr benzersiz değil!'
assert df_etken_madde_nihai['yol'].value_counts().sum() == total_etken, f'HATA: Yol toplamı total_etken ({total_etken}) ile uyuşmuyor!'
assert (df_etken_madde_nihai['ddinter_id'].notna() == df_etken_madde_nihai['eslesme_kaynagi'].notna()).all(), 'HATA: ddinter_id ile eslesme_kaynagi tutarsız!'
assert 'atc_ana' in df_etken_madde_nihai.columns, 'HATA: atc_ana sütunu eksik!'

rapor_metni = "\n".join(rapor_lines) + "\n"
rapor_yol = kok / 'docs' / 'esleme_raporu.md'
rapor_yol.write_text(rapor_metni, encoding='utf-8')

print(f"✓ docs/esleme_raporu.md başarıyla oluşturuldu: {rapor_yol}")
print("✓ Tüm bütünlük assertion kontrolleri başarıyla geçti.")
print()
print("Rapor Başlığı ve Özet Cümle:")
print(rapor_lines[2])
print(rapor_lines[3])


✓ docs/esleme_raporu.md başarıyla oluşturuldu: /Users/reyhanbuseyanar/Desktop/ddi-final/docs/esleme_raporu.md
✓ Tüm bütünlük assertion kontrolleri başarıyla geçti.

Rapor Başlığı ve Özet Cümle:
> **Özet:** Türkiye'deki kapsam içi etken maddelerin %46.1'i (1,378 / 2,988), kapsam içi ruhsatlı ürünlerin %76.5'inin (17,180 / 22,470) en az bir bileşeni DDInter'da bulunuyor.
> *(Tüm Bileşenler Bazında: %44.3 (1,378 / 3,111) — Tüm Ruhsatlı Ürünler Bazında: %76.7 (17,578 / 22,927))*


---
## 15. İP3 Teslimat Özeti, Kritik Bulgular ve Teknik Değerlendirme

Bu çalışma kitabı ile İP3 (İngilizce Ad Köprüsü ve DDInter Eşlemesi) adımları uygulanmış ve sayısal hedeflere ulaşılmıştır:
- **Kapsam İçi Etken Madde DDInter Eşleşme Oranı:** %46.1 (1,378 / 2,988)
- **Kapsam İçi Ruhsatlı Ürün Kapsama Oranı (en az 1 bileşeni DDInter'da olan):** %76.5 (17,180 / 22,470) — *Hedeflenen %70 eşiği aşılmıştır.*
- **Tüm Bileşenler Bazında:** %44.3 (1,378 / 3,111)
- **Tüm Ruhsatlı Ürünler Bazında:** %76.7 (17,578 / 22,927)

### ⚠️ Rapor ve Sonraki İş Paketleri (İP4 / D05 / E09) İçin Kritik Teknik Kısıtlar

Sayısal başarı oranlarının arkasında, veri kümesinin doğasından ve upstream adımlardan kaynaklanan ve koşulsuz başarı iddiasını sınırlandıran 5 temel gerçek bulunmaktadır:

1. **Eşleşme Oranı $\neq$ Etkileşim Verisi Kapsamı (DDInter Grup Asimetrisi):**
   DDInter indirme paketlerinde yalnızca 8 ana grup (A, B, D, H, L, P, R, V) mevcuttur; Türkiye'de en yaygın reçetelenen **C (Kardiyovasküler), G (Üro-genital), J (Antibiyotikler), M (Kas-İskelet), N (Sinir Sistemi) ve S (Duyu)** gruplarının müstakil etkileşim dosyaları bulunmamaktadır. Dolayısıyla N grubunda %69.8 eşleşme sağlanmış olması, N ilaçlarının etkileşim verisinin tam olduğu anlamına **gelmez**. Bu ilaçlar yalnızca mevcut 8 gruptaki partnerleriyle olan etkileşimlerde yer alabilir; iki N veya iki C ilacının kendi arasındaki etkileşimi DDInter'da tespit edilemez. E09 grafiği ve final raporunda bu kısıt açıkça vurgulanmalıdır.

2. **1.378 Eşleşen Kayıt $\rightarrow$ 852 Benzersiz İlaç (Molekül Büyüklüğü):**
   Eşleşen 1.378 bileşen satırı, DDInter kütüphanesinde yalnızca **852 benzersiz ilaç kimliğine** (`ddinter_id`) karşılık gelmektedir. Aradaki 526 satırlık fark; Türkçe yazım varyantlarından (`insulin lispro` formları, aşı adları), tuz/ester ayrımlarından ve ayrıştırma farklılıklarından kaynaklanmaktadır.

3. **Yanlış Birleşme Tespiti ve İP2 Kaynağı (Kişi A Geri Bildirimi):**
   Tekil kimlik denetimlerinde, `ketokonazol` (antifungal) bileşeninin, TEKNAZOL tabletlerin `urun.csv` dosyasında hatalı olarak `M01AE03` (ketoprofen) kodlanması sebebiyle Yol A'da ketoprofen ATC'sine bağlanarak `DDInter1009` (ketoprofen) kimliğiyle yanlış birleştiği tespit edilmiştir. Bu ve benzeri yazım çeşitlilikleri Kişi A'nın İP2 ayrıştırma ve veri temizliği tarafına iletilmiştir.

4. **Yol C'nin Katkısı ve DDInter Kapsam Sınırı:**
   Yol C kural motorunun %52.8'lik doğruluğu Yol A çiftleri (eğitim kümesi) üzerinde ölçüldüğü için iyimserdir. Kalan ~1.000 bileşenden yalnızca 78'i DDInter'a bağlanabilmiştir. Buradaki temel darboğaz çeviri kuralı eksikliği değil, DDInter'ın 1.939 bileşiklik genel kapsamıdır. Eşleşmeyenlerin başında gelen `dekstroz`, `kalsiyum klorur`, `sodyum laktat`, `magnezyum klorur`, `tiyokolsikosid` ve `deksketoprofen` gibi bileşikler DDInter veritabanında hiç yer almamaktadır; dolayısıyla ek çeviri kuralları bu sayıyı artırmayacaktır.

5. **SKRS Karakter Düzeltmeleri ve Kural 6:**
   SKRS adlarındaki noktasız `ı` karakter hataları (`combınatıon` $\rightarrow$ `combination`) düzeltilmiş, `paracetamol` PubChem yayılımı ile `acetaminophen`'e bağlanmıştır. Ancak `allergen extracts`, `peritoneal dialytics` gibi ATC sınıf adları ile Kural 6 ürünlerinin takibi için Kişi A ve Kişi C ile koordinasyon sürmektedir.

**Tamamlanan Altyapı Adımları:**
- [x] **SKRS ATC Sözlüğü:** Resmi sözlük aktif+pasif sekmelerinden çıkarıldı.
- [x] **DDInter İlaç Tablosu:** `ddinter_ilac.csv` sayısal ID sıralamasıyla oluşturuldu (1.939 bileşik).
- [x] **Yol A:** Tek bileşenli, 7 haneli ATC'li bileşenler doğrudan ATC sözlüğünden eşleştirildi (1.922 bileşen, Kişi A).
- [x] **Yol B:** Kombine ürünlerde ATC adından ayrıştırılan bileşenler RapidFuzz ile eşleştirildi (66 bileşen, Kişi A).
- [x] **SKRS Adımı:** SKRS kaynaklı ürünlerin bileşenleri `ı` temizliğiyle İngilizce olarak işlendi (90 bileşen, Kişi B).
- [x] **Yol C:** `config/tr_en_kurallar.csv` 9 ölçülmüş kural motoruyla kalan bileşenler bağlandı (78 bileşen, Kişi B).
- [x] **PubChem Kontrolü:** Rate limit, disk cache ve `ad_en_ham` izlenebilirliği ile en sık 100 eşleşmeyenden 16 bileşen kurtarıldı ve 24 satıra yayıldı (Kişi B).
- [x] **Yol D Taslağı:** Kişi C için `atc_ipucu`, `urun_ornegi` ve `kapsam_ici` öncelikli `data/interim/ruhsatli_esleme_taslagi.csv` çıkarıldı (Kişi B).
- [x] **Nihai Tablo:** `atc_ana` sütunu eklenmiş `data/interim/etken_madde.csv` kaydedildi (3.111 satır, Kişi B).
- [x] **Eşleşme Raporu:** 4 zorunlu tablo, bütünlük assertion'ları ve kısıt notlarıyla `docs/esleme_raporu.md` üretildi (Kişi B).
